# Steam Games: Data Preparation for Interactive Visualization

## Abstract
This notebook documents the data preparation and transformation procedures used to support an interactive Steam analytics dashboard developed with Plotly. The analysis focuses on Metacritic scores, release years, game prices, genres, and Steam user reviews. Its purpose is to describe the analytical decisions, provide reproducible transformations, and organize datasets suitable for subsequent visualizations.

The original source contained approximately **89,618 records**. The academic project selected games with available Metacritic scores, yielding **3,527 records** in the prepared dataset used here. Consequently, the findings describe this **filtered sample**, not the complete Steam catalog.

> **Scope:** This notebook prepares and inspects data. Plotly chart generation and the HTML dashboard are implemented separately in `main.py`.

## Research questions and analytical approach

1. **Temporal trends:** How has the average Metacritic score varied by release year? → Aggregate scores by year.
2. **Genre comparison:** Which genres have higher mean Metacritic scores? → Normalize multi-label genres and calculate genre-level averages.
3. **Audience reception:** How are positive and negative Steam reviews distributed among genres? → Sum review counts by genre and reshape to long format.
4. **Price and critical reception:** Is there an observable relationship between price and Metacritic score? → Prepare game-level observations for a scatter plot.

These analyses are **descriptive and exploratory**. Differences and visual associations do not establish causal relationships.

## 1. Libraries and data requirements

**Pandas** is used for tabular data manipulation, grouping, aggregation, and restructuring. Python's standard `ast` module is used to interpret genre lists stored as text.

The notebook expects a previously cleaned CSV file at `data/metacritic-steam.csv`. Because it is an input dataset, the earlier filtering and feature-engineering steps are not repeated here. For reproducibility, run the notebook from the repository's root directory.

In [1]:
import pandas as pd
import ast

## 2. Parsing and standardizing genre labels

The `genres` column may contain list-like information serialized as strings (for example, `"['Action', 'RPG']"`). Before aggregating by genre, these values must be converted into consistent Python lists.

The `convert_genres()` helper function:

- Handles missing values by returning an empty list.
- Safely attempts to parse a string representation of a list using `ast.literal_eval` (rather than `eval`).
- Accepts a single genre string as a one-element list.
- Removes whitespace, blank entries, and duplicate labels **within each game**.

**Methodological note:** Invalid or unrecognized values become empty lists. Such rows will not contribute to genre-based analyses, although they may remain in the overall dataset.

In [2]:
def convert_genres(value):
    if pd.isna(value): return []

    if isinstance(value, list): genres = value
    
    else:
        try:
            genres = ast.literal_eval(value)
        except (ValueError, SyntaxError, TypeError):
            return []

    # Ensures that the result is always a list
    if isinstance(genres, str):
        genres = [genres]

    if not isinstance(genres, (list, tuple, set)):
        return []

    # Remove extra spaces, blank values, and duplicates per game
    genres = [
        genre.strip()
        for genre in genres
        if isinstance(genre, str) and genre.strip()
    ]

    return list(dict.fromkeys(genres))

## 3. Loading and inspecting the prepared dataset

The `read_csv()` function imports the filtered CSV into a Pandas DataFrame. A preliminary inspection confirms the number of observations, identifies the available fields, and checks whether variables required by the analysis contain missing values.

At this stage, the expected variables include `year`, `metacritic_score`, `genres`, `positive`, `negative`, `price`, and `name`. Variable definitions should be interpreted according to the dataset documentation.

In [3]:
file_path = 'data/metacritic-steam.csv'
df = pd.read_csv(file_path)
df

,Unnamed: 0,appid,name,release_date,required_age,price,dlc_count,windows,mac,linux,...,negative,discount,peak_ccu,num_reviews_total,num_reviews_recent,year,dev_count,pub_count,screenshot_count,movie_count
0,2,570,Dota 2,2013-07-09,0,0.00,2,True,True,True,...,451338,0,555977,2452595,29366,2013,1,1,10,5
1,3,271590,Grand Theft Auto V Legacy,2015-04-13,17,0.00,0,True,False,False,...,250012,0,117698,1803832,17517,2015,1,1,76,32
2,5,440,Team Fortress 2,2007-10-10,0,0.00,1,True,False,True,...,120619,0,50817,1146642,8172,2007,1,1,15,17
3,6,105600,Terraria,2011-05-16,0,9.99,2,True,True,True,...,34460,0,30516,1102434,8860,2011,1,1,19,3
4,7,252490,Rust,2018-02-08,0,39.99,4,True,True,False,...,152272,0,200902,993856,14960,2018,1,1,32,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3522,80738,42900,Hearts of Iron III: Semper Fi,2010-06-07,0,4.99,3,True,False,False,...,0,0,0,-1,-1,2010,1,1,9,0
3523,84571,924130,Slime-san: Creator,2018-12-08,0,5.99,0,True,True,True,...,0,0,0,-1,-1,2018,2,1,5,1
3524,85360,667060,No Escape,2017-12-13,0,5.99,0,True,False,False,...,3,0,0,-1,-1,2017,1,1,13,3
3525,87477,1213050,Ink,2021-02-03,0,10.99,0,True,False,False,...,0,0,0,-1,-1,2021,1,1,12,3


## 4. Average Metacritic score by release year

A yearly mean summarizes critical ratings for games released during the same calendar year. It supports the study of changes in average scores across the available time series.

The operation consists of grouping records by `year` and calculating the arithmetic mean of `metacritic_score` for each group:

**Interpretation caution:** Years with few scored games may have unstable averages. Observed differences can also reflect changes in sample coverage rather than changes in game quality.

In [4]:
df_yearscore = df.groupby('year', as_index=False).agg(
    {'metacritic_score':'mean'}
)

df_yearscore

,year,metacritic_score
0,1997,56.000000
1,2001,81.000000
2,2002,88.000000
3,2004,88.000000
4,2005,81.000000
5,2006,79.666667
6,2007,79.600000
7,2008,84.666667
8,2009,73.258065
9,2010,72.439024


## 5. Genre normalization and average score by genre

A game can belong to multiple genres. This creates a **many-to-many** analytical relationship: one game may contribute one observation to each associated genre.

First, `apply(convert_genres)` standardizes each record's genre values. Next, `explode('genres')` expands each list into separate rows while retaining the game's original attributes. Empty genre values are excluded from the genre-level analysis.

Finally, `groupby('genres')` calculates the average `metacritic_score` for each category. Sorting these values prepares a readable horizontal bar chart.

**Interpretation caution:** Genre averages are not mutually exclusive because the same game may be represented in several genres. Also, a genre with few games can have a high average without providing strong evidence of a general pattern.

In [5]:
# Applying Converting Genres into the main DataFrame
df['genres'] = df['genres'].apply(convert_genres)

# Separating all values from the list without losing the game statistics
df_genres = (
    df
    .explode('genres')
    .dropna(subset=['genres'])
    .copy()
)

# Creating Average Metacritic Score by Genre DataFrame
df_genrescore = df_genres.groupby('genres', as_index=False).agg(
    {'metacritic_score':'mean'}
)
df_genrescore.sort_values(by='metacritic_score',inplace=True)

df_genrescore

,genres,metacritic_score
5,Gore,58.500000
15,Violent,66.500000
3,Early Access,68.500000
10,Racing,71.924528
12,Simulation,72.049822
13,Sports,72.444444
6,Indie,72.931002
0,Action,73.042759
14,Strategy,73.187500
2,Casual,73.469185


## 6. Aggregating Steam reviews by genre

The `positive` and `negative` variables represent review counts. To summarize player reception within each genre, counts are added across all associated games using `groupby()` and `sum()`.

This approach measures the **volume of reviews**, not the percentage of favorable opinions or the number of unique reviewers. Genres with more games or more popular titles may naturally accumulate more reviews.

**Important limitation:** Since games can belong to multiple genres, the same game's review counts can appear in multiple genre totals. Accordingly, adding review totals across genres would double-count some reviews and should not be interpreted as the platform-wide total.

In [6]:
# Creating sub dataset to count genre positive and negative review
df_genrereview = (
    df_genres
    .groupby('genres', as_index=False).agg(
        {
            'positive':'sum',
            'negative':'sum'
        }
    )
)

# Creating Positive and Negative Reviews by Genre Order
df_genreorder = (
    df_genrereview
    .sort_values('positive', ascending=True)['genres']
    .tolist()
)

df_genreorder

['Sexual Content',
 'Nudity',
 'Early Access',
 'Gore',
 'Violent',
 'Racing',
 'Sports',
 'Casual',
 'Massively Multiplayer',
 'Free To Play',
 'Simulation',
 'Strategy',
 'RPG',
 'Indie',
 'Adventure',
 'Action']

### 6.1 Reshaping review data to long format

The review aggregation initially produces two measurement columns: `positive` and `negative`. For a stacked bar chart, Plotly Express works conveniently with a **long-form** structure in which each row identifies a genre, review type, and review count.

The `melt()` operation converts these two columns into:

- `genres` — the genre identifier;
- `review type` — whether the count is positive or negative;
- `quantity` — the corresponding number of reviews.

This transformation changes the *shape*, not the numeric content, of the aggregated data.

In [7]:
# Creating Positive and Negative Reviews by Genre DataFrame
df_genrereview = df_genrereview.melt(
    id_vars='genres',
    value_vars=['positive', 'negative'],
    var_name='review type',
    value_name='quantity'
)

df_genrereview['review type'] = (
    df_genrereview['review type']
)

df_genrereview

,genres,review type,quantity
0,Action,positive,38225809
1,Adventure,positive,24466082
2,Casual,positive,4065602
3,Early Access,positive,2690
4,Free To Play,positive,6982998
5,Gore,positive,8463
6,Indie,positive,21736472
7,Massively Multiplayer,positive,4382779
8,Nudity,positive,252
9,RPG,positive,18508802


## 7. Preparing game-level observations for price–score analysis

The price-versus-score scatter plot requires one observation per game, with `price` on the horizontal axis and `metacritic_score` on the vertical axis. Game names and release years are retained to provide contextual information in interactive tooltips.

Only the variables relevant to the plot are selected. The data are sorted by score for inspection, and `.copy()` creates an independent DataFrame.

**Interpretation caution:** A scatter plot can reveal patterns, concentration, or outliers, but it cannot determine whether a higher price *causes* a better review score. Prices may depend on age, publisher strategy, regional pricing, discounts, and other factors.

In [8]:
columns = ['name','price','metacritic_score','year']
df_pricescore = df[columns].sort_values('metacritic_score',ascending=False).copy()
df_pricescore


,name,price,metacritic_score,year
102,Disco Elysium - The Final Cut,39.99,97,2019
10,Baldur's Gate 3,59.99,96,2023
1,Grand Theft Auto V Legacy,0.00,96,2015
49,The Elder Scrolls V: Skyrim,19.99,94,2011
94,BioShock Infinite,29.99,94,2013
...,...,...,...,...
3310,Majestic Nights,4.99,32,2014
3172,Shaq Fu: A Legend Reborn,19.99,30,2018
2348,Wild West Online,0.00,29,2018
1124,FlatOut 3: Chaos & Destruction,7.99,23,2011


## Reproducibility and limitations

The transformations generate four principal datasets for the Plotly dashboard:

| DataFrame | Unit of analysis | Intended visualization |
|---|---|---|
| `df_yearscore` | Release year | Line chart of average Metacritic score |
| `df_genrescore` | Genre | Horizontal bar chart of mean score |
| `df_genrereview` | Genre × review type | Stacked horizontal review-count chart |
| `df_pricescore` | Game | Price-versus-score scatter plot |

**Scope and limitations:**

1. **Selection bias:** The prepared sample includes only games with a Metacritic score and is not representative of every Steam game.
2. **Genre overlap:** Multi-genre games contribute to multiple category summaries; genre totals are not additive across categories.
3. **Unequal group sizes:** Release years and genres differ in sample size, which affects the stability of comparisons.
4. **Descriptive analysis:** Trends and correlations must not be presented as evidence of causation.
5. **Data currency:** Results reflect the version of the downloaded and filtered data, not live Steam or Metacritic metrics.

## Conclusion

The notebook converts a prepared Steam dataset into consistent, visualization-ready analytical tables. By documenting parsing, aggregation, reshaping, and relevant methodological limitations, it provides a transparent foundation for the interactive Plotly dashboard and the interpretation of its results.

**Academic context:** Pontifical Catholic University of Rio Grande do Sul (PUCRS), Data Visualization course.